In [1]:
from Scripts.Random_forest_nested import *
import pickle
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel('../SImulated_data/Clinical_synthetic.xlsx', index_col=0)
X,y=df.loc[:,['scaled_nox', 'scaled_no2', 'scaled_formaldehyd', 'scaled_acetone',
       'scaled_acetald', 'scaled_bc', 'scaled_PM25', 'no2_street', 'nox_street',
       'pm25_street', 'bmi_6_yrs', 'gestational_age', 'mother\'s_education_1yr',
        'genetic_risk_score_asthma', 'PCA1', 'PCA2']],df.asthma_0_7yrs
NUM_TRIALS=2

In [3]:
list_parameters=["$NO_x$","$NO_2$","Formaldehyd","Acetone","Acetald","Black Carbon","$PM_{2.5}$","Street $NO_2$","Street $NO_x$","Street $PM_{2.5}$","BMI","Gestational age","Mother Education","Asthma risk score","Landuse 1","Landuse 2"]


In [4]:
replace_dict=dict(zip(X.columns,list_parameters))
X=X.rename(columns=replace_dict)

In [5]:
param_distributions = {
    'n_estimators':           [300],
    'max_depth':              randint(2, 6),
    'min_samples_leaf':       randint(10, 30),
    'max_features':           [0.4, 0.5, 0.6, 'sqrt'],
    'max_samples':            [0.5, 0.6, 0.7, 0.8],
    # class_weight removed
}
dataframe=nested_cross_calibrated_rf(
    X, y,
    Number_CV=2,
    number_split=3,
    n_neighbors_imputer=25,
    param_distributions=param_distributions,
    calibration_method="isotonic")

Fold 1/6  AUC=0.598  BSS=-0.015  ICI=0.048
Fold 2/6  AUC=0.540  BSS=-0.001  ICI=0.027
Fold 3/6  AUC=0.632  BSS=+0.035  ICI=0.021
Fold 4/6  AUC=0.558  BSS=+0.005  ICI=0.022
Fold 5/6  AUC=0.556  BSS=-0.022  ICI=0.048
Fold 6/6  AUC=0.475  BSS=-0.049  ICI=0.054


In [8]:
with open("../Performence/results_rf_7y_adjusted.pkl", "wb") as f:
    pickle.dump(dataframe, f)

In [ ]:
fig=shap.plots.violin(dataframe['shap_values_oof'],X ,plot_type="layered_violin",show=False,max_display=30)
plt.savefig('Plots/shap_7_adjusted.png',dpi=300,bbox_inches='tight')
